In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gc
sns.set_theme()
gc.enable()

In [2]:
pos_data=pd.read_csv("POS_CASH_balance.csv")
pos_data.shape

(10001358, 8)

In [3]:
pos_data.head()

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0,45.0,Active,0,0
1,1715348,367990,-33,36.0,35.0,Active,0,0
2,1784872,397406,-32,12.0,9.0,Active,0,0
3,1903291,269225,-35,48.0,42.0,Active,0,0
4,2341044,334279,-35,36.0,35.0,Active,0,0


In [32]:
pos_data.isnull().sum()

SK_ID_PREV                   0
SK_ID_CURR                   0
MONTHS_BALANCE               0
CNT_INSTALMENT           26071
CNT_INSTALMENT_FUTURE    26087
NAME_CONTRACT_STATUS         0
SK_DPD                       0
SK_DPD_DEF                   0
dtype: int64

In [11]:
'''
curr_prev_table stores the unique mapping of previous loans to current loans as a dataframe
'''
curr_prev_table=pos_data[['SK_ID_CURR', 'SK_ID_PREV']].drop_duplicates()
curr_prev_table.head()

,SK_ID_CURR,SK_ID_PREV
0,182943,1803195
1,367990,1715348
2,397406,1784872
3,269225,1903291
4,334279,2341044


### Aggregation Based on SK_ID_PREV

In [33]:
group=pos_data.groupby(by='SK_ID_PREV')

In [54]:
pos_agg=group.agg(func={'CNT_INSTALMENT':  ['min', 'max'], 'SK_ID_PREV': 'count', 
                'SK_DPD': 'max', 'SK_DPD_DEF': 'max'})
pos_agg.head()

CNT_INSTALMENT       SK_ID_PREV SK_DPD SK_DPD_DEF
                      min   max      count    max        max
SK_ID_PREV                                                  
1000001               2.0  12.0          3      0          0
1000002               4.0   6.0          5      0          0
1000003              12.0  12.0          4      0          0
1000004               7.0  10.0          8      0          0
1000005              10.0  10.0         11      0          0

In [55]:
pos_agg.shape

(936325, 5)

In [56]:
'''
renaming columns appropriatley and flattent the dataframe
'''
pos_agg.columns=pd.Index(['_'.join(val) for val in pos_agg.columns.to_flat_index()])
pos_agg.reset_index(inplace=True)
pos_agg.head()

,SK_ID_PREV,CNT_INSTALMENT_min,CNT_INSTALMENT_max,SK_ID_PREV_count,SK_DPD_max,SK_DPD_DEF_max
0,1000001,2.0,12.0,3,0,0
1,1000002,4.0,6.0,5,0,0
2,1000003,12.0,12.0,4,0,0
3,1000004,7.0,10.0,8,0,0
4,1000005,10.0,10.0,11,0,0


In [59]:
'''
adding INSTALMENT_REDUCTION_PCT
'''
pos_agg['INSTALMENT_REDUCTION_PCT']=(pos_agg['CNT_INSTALMENT_max']-pos_agg['CNT_INSTALMENT_min'])*100/pos_agg['CNT_INSTALMENT_max']
pos_agg.head()

,SK_ID_PREV,CNT_INSTALMENT_min,CNT_INSTALMENT_max,SK_ID_PREV_count,SK_DPD_max,SK_DPD_DEF_max,INSTALMENT_REDUCTION_PCT
0,1000001,2.0,12.0,3,0,0,83.333333
1,1000002,4.0,6.0,5,0,0,33.333333
2,1000003,12.0,12.0,4,0,0,0.000000
3,1000004,7.0,10.0,8,0,0,30.000000
4,1000005,10.0,10.0,11,0,0,0.000000


#### Aggregating MONTH_BALANCE AND NAME_CONTRACT_STATUS

In [66]:
'''
temp will contain the indices of the max month balance corresponding to each previous loan (recent most entry).
the indices will be according to the main data(pos_data). Using temp as index the rows can be accessed that tells the recent most
entry for the previous loan. That row is used to extract the recent most status of the contract
Lastly, data is merged to update pos_agg to obtain the final table
'''
temp=group['MONTHS_BALANCE'].idxmax()
temp2=pos_data.loc[temp, ['SK_ID_PREV', 'SK_ID_CURR', 'MONTHS_BALANCE', 'NAME_CONTRACT_STATUS']]
pos_agg=pd.merge(left=temp2, right=pos_agg, on='SK_ID_PREV')
pos_agg.head()

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,NAME_CONTRACT_STATUS,CNT_INSTALMENT_min,CNT_INSTALMENT_max,SK_ID_PREV_count,SK_DPD_max,SK_DPD_DEF_max,INSTALMENT_REDUCTION_PCT
0,1000001,158271,-8,Completed,2.0,12.0,3,0,0,83.333333
1,1000002,101962,-50,Completed,4.0,6.0,5,0,0,33.333333
2,1000003,252457,-1,Active,12.0,12.0,4,0,0,0.000000
3,1000004,260094,-22,Completed,7.0,10.0,8,0,0,30.000000
4,1000005,176456,-46,Completed,10.0,10.0,11,0,0,0.000000


In [68]:
pos_agg.shape

(936325, 10)